# 04 · Model Evaluation


Comprehensive evaluation of all five trained models on the 2024-2025 holdout 
test set. Classification models (Logistic Regression, Random Forest, XGBoost, 
LightGBM) are assessed with classification metrics, confusion matrices, and 
feature importance. Linear Regression is assessed with regression metrics and 
threshold-based classification for comparison parity.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score, roc_auc_score,
)

DATA_DIR = Path("../../data_pipeline/data/processed")
MODEL_DIR = Path("../models")

# Load artifacts
X_test = pd.read_csv(DATA_DIR / "X_test.csv")
y_test = pd.read_csv(DATA_DIR / "y_test.csv")["target_risk"]
scaler = joblib.load(MODEL_DIR / "scaler.joblib")
feature_cols = json.load(open(MODEL_DIR / "feature_columns.json")) if (MODEL_DIR / "feature_columns.json").exists() else None

# Load all trained models
models = {}
for name in ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM", "Linear Regression"]:
    safe_name = name.lower().replace(" ", "_")
    path = MODEL_DIR / f"{safe_name}.joblib"
    if path.exists():
        models[name] = joblib.load(path)
    else:
        print(f"Warning: {name} model not found at {path}")

print(f"Loaded {len(models)} models")

# ─── Detailed Classification Report ───
for name, model in models.items():
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")

    if name == "Linear Regression":
        y_pred_proba = np.clip(model.predict(scaler.transform(X_test)), 0, 1)
        y_pred = (y_pred_proba >= 0.5).astype(int)
        print(classification_report(y_test, y_pred, target_names=["Safe (0)", "Risk (1)"]))
        cm = confusion_matrix(y_test, y_pred)
    else:
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        print(classification_report(y_test, y_pred, target_names=["Safe (0)", "Risk (1)"]))
        cm = confusion_matrix(y_test, y_pred)
        print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

    # Confusion matrix heatmap
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Safe (0)", "Risk (1)"],
                yticklabels=["Safe (0)", "Risk (1)"])
    plt.title(f"{name} — Confusion Matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.savefig(MODEL_DIR / f"cm_{name.lower().replace(' ', '_')}.png", dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  💾 Confusion matrix saved")

# ─── ROC Curves ───
plt.figure(figsize=(10, 8))
for name, model in models.items():
    if name == "Linear Regression":
        y_pred_proba = np.clip(model.predict(scaler.transform(X_test)), 0, 1)
    else:
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], "k--", alpha=0.5)
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves — All Models", fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / "roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n💾 ROC curves saved to models/roc_curves.png")

# ─── Precision-Recall Curves ───
plt.figure(figsize=(10, 8))
for name, model in models.items():
    if name == "Linear Regression":
        y_pred_proba = np.clip(model.predict(scaler.transform(X_test)), 0, 1)
    else:
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
    ap = average_precision_score(y_test, y_pred_proba)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})", linewidth=2)

plt.xlabel("Recall", fontsize=12)
plt.ylabel("Precision", fontsize=12)
plt.title("Precision-Recall Curves — All Models", fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODEL_DIR / "precision_recall_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("💾 Precision-Recall curves saved to models/precision_recall_curves.png")

# ─── Summary Table ───
print("\n" + "=" * 80)
print("SUMMARY TABLE")
print("=" * 80)
summary_rows = []
for name, model in models.items():
    if name == "Linear Regression":
        y_pred_proba = np.clip(model.predict(scaler.transform(X_test)), 0, 1)
        y_pred = (y_pred_proba >= 0.5).astype(int)
    else:
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

    summary_rows.append({
        "Model": name,
        "Accuracy": float((y_pred == y_test).mean()),
        "Precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "F1": float(f1_score(y_test, y_pred, zero_division=0)),
        "ROC-AUC": float(roc_auc_score(y_test, y_pred_proba)),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("Recall", ascending=False)
print(summary_df.round(4).to_string(index=False))
summary_df.to_csv(MODEL_DIR / "model_summary.csv", index=False)
print(f"\n💾 Summary saved to models/model_summary.csv")
